# SolubilityAI — Molecular Machine Learning for Aqueous Solubility

**Goal:** Predict experimental aqueous solubility (logS) from molecular SMILES using 2,048-bit Morgan fingerprints and Random Forest regression.

**Dataset:** [AqSolDB curated CSV](https://github.com/semmatoninn/AqSolDB) (9,982 source records; 9,980 cleaned in our original run). **Run all cells from top to bottom** in Google Colab using a CPU runtime. Generated outputs are saved under `outputs/` in the current runtime; download them if you want to keep them.

**Previously observed results** (from the author's original Colab experiment; not newly recomputed by this cleaned notebook): random split MAE **1.1357**, RMSE **1.5112**, R² **0.6108**; scaffold-group split MAE **1.4686**, RMSE **1.8331**, R² **0.5091**. Re-running can produce slight differences across dependency versions.

> Research demonstration only. Solubility measurements depend on experimental conditions, molecular standardization is limited, and predictions are not validated formulation advice.

## 1. Environment and setup
Install the Python dependencies. No GPU is needed. The notebook does not require Google Drive mounting.

In [ ]:
%pip -q install rdkit pandas numpy scikit-learn matplotlib joblib


In [ ]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RDLogger.DisableLog('rdApp.warning')  # Suppress repeated warnings; errors are still shown
OUT = Path('outputs')
OUT.mkdir(exist_ok=True)
SEED = 42


## 2. Download and clean AqSolDB
Read the original curated CSV directly, validate SMILES with RDKit, canonicalize them, and average repeated measurements of the same canonicalized structure. This is a basic cleaning approach, **not** comprehensive salt/tautomer/assay-condition standardization.

In [ ]:
URL = ('https://raw.githubusercontent.com/semmatoninn/AqSolDB/main/'
       'curated-solubility-dataset.csv')
raw = pd.read_csv(URL)
print('Original rows/columns:', raw.shape)
assert {'SMILES', 'Solubility'}.issubset(raw.columns)

clean = raw[['SMILES', 'Solubility']].copy()
clean['Solubility'] = pd.to_numeric(clean['Solubility'], errors='coerce')
clean = clean.dropna(subset=['SMILES', 'Solubility'])
clean['mol'] = clean['SMILES'].apply(Chem.MolFromSmiles)
invalid = int(clean['mol'].isna().sum())
clean = clean.loc[clean['mol'].notna()].copy()
clean['smiles'] = clean['mol'].apply(Chem.MolToSmiles)
df = (clean.groupby('smiles', as_index=False)['Solubility']
      .mean().rename(columns={'Solubility': 'Y'}))
print('Invalid structures:', invalid)
print('Cleaned unique structures:', len(df))
display(df.head())
df.to_csv(OUT / 'aqsoldb_clean.csv', index=False)


### Solubility distribution
Interpret the left tail with care: very low logS compounds are comparatively uncommon. The histogram is not perfectly symmetric.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(df['Y'], bins=50, color='royalblue', edgecolor='white')
ax.set(xlabel='Experimental aqueous solubility (logS)', ylabel='Number of molecules',
       title='AqSolDB solubility distribution')
fig.tight_layout()
fig.savefig(OUT / 'solubility_distribution.png', dpi=180)
plt.show()
display(df['Y'].describe())


## 3. Molecular fingerprints
Morgan radius 2 fingerprints encode local neighborhoods into hashed 2,048-bit binary vectors. The arrays `X` and `y` retain the same row order.

In [ ]:
fpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
molecules = [Chem.MolFromSmiles(s) for s in df['smiles']]
assert all(m is not None for m in molecules)
X = np.stack([fpgen.GetFingerprintAsNumPy(m) for m in molecules]).astype(np.uint8)
y = df['Y'].to_numpy(dtype=float)
assert X.shape == (len(df), 2048) and len(y) == len(df)
print('X:', X.shape, '| y:', y.shape, '| first active bits:', int(X[0].sum()))


In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.imshow(X[:15, :100], cmap='Blues', interpolation='nearest', aspect='auto', vmin=0, vmax=1)
ax.set(xlabel='Fingerprint bit position', ylabel='Molecule index',
       title='First 100 bits of the first 15 fingerprints')
fig.tight_layout()
fig.savefig(OUT / 'fingerprint_heatmap.png', dpi=180)
plt.show()


## 4. Random-split baseline and Random Forest
A reproducible 80/20 random split provides the initial benchmark. Evaluate both a constant-mean predictor and a 200-tree Random Forest using identical held-out molecules.

In [ ]:
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, np.arange(len(y)), test_size=0.20, random_state=SEED)
baseline = DummyRegressor(strategy='mean').fit(X_train, y_train)
baseline_predictions = baseline.predict(X_test)
rf = RandomForestRegressor(n_estimators=200, max_features='sqrt',
    min_samples_leaf=2, random_state=SEED, n_jobs=-1)
t0 = time.time()
rf.fit(X_train, y_train)
rf_predictions = rf.predict(X_test)
print('Random-split training seconds:', round(time.time()-t0, 1))

def metrics(name, actual, predicted):
    return {'Model': name, 'MAE': mean_absolute_error(actual, predicted),
            'RMSE': np.sqrt(mean_squared_error(actual, predicted)),
            'R2': r2_score(actual, predicted)}

random_metrics = pd.DataFrame([
    metrics('Mean baseline', y_test, baseline_predictions),
    metrics('Random Forest', y_test, rf_predictions)])
display(random_metrics.round(4))
random_metrics.to_csv(OUT / 'random_split_metrics.csv', index=False)


### Predicted versus observed
The dashed line indicates perfect prediction. Points above it overpredict logS; points below underpredict. Large errors at both extremes are worth examining.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))
ax.scatter(y_test, rf_predictions, s=12, alpha=.30)
limits = [min(y_test.min(), rf_predictions.min()), max(y_test.max(), rf_predictions.max())]
ax.plot(limits, limits, 'r--', label='Perfect prediction')
ax.set(xlabel='Experimental logS', ylabel='Predicted logS',
       title='Random split — predicted vs experimental')
ax.legend(); fig.tight_layout()
fig.savefig(OUT / 'random_prediction_scatter.png', dpi=180)
plt.show()


### Error breakdown by measured solubility
These three bins are exploratory analysis categories, not validated pharmaceutical thresholds.

In [ ]:
err = pd.DataFrame({'Experimental': y_test, 'Predicted': rf_predictions})
err['Signed_Error'] = err['Predicted'] - err['Experimental']
err['Absolute_Error'] = err['Signed_Error'].abs()
err['Solubility_Range'] = pd.cut(err['Experimental'],
    [-np.inf, -6, -2, np.inf], labels=['Low (< -6)', 'Medium (-6 to -2)', 'High (> -2)'])
error_summary = err.groupby('Solubility_Range', observed=True).agg(
    Molecules=('Signed_Error', 'size'), MAE=('Absolute_Error', 'mean'),
    Mean_Error=('Signed_Error', 'mean'))
display(error_summary.round(3))
error_summary.to_csv(OUT / 'solubility_range_errors.csv')
ax = error_summary['MAE'].plot.bar(figsize=(7, 4), rot=0, title='Random split — MAE by solubility range')
ax.set_ylabel('MAE (logS)'); plt.tight_layout()
plt.savefig(OUT / 'solubility_range_errors.png', dpi=180); plt.show()


## 5. Bemis–Murcko scaffold-group evaluation
Extract ring/linker scaffold groups and split *groups* rather than individual molecules. No identical scaffold group appears in both partitions. `test_size=0.20` targets 20% of the **groups**, not 20% of molecules; acyclic molecules may share an empty scaffold identifier.

In [ ]:
scaffolds = np.array([MurckoScaffold.MurckoScaffoldSmiles(mol=m) for m in molecules])
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
sc_train_idx, sc_test_idx = next(gss.split(X, y, groups=scaffolds))
assert set(scaffolds[sc_train_idx]).isdisjoint(set(scaffolds[sc_test_idx]))
print('Scaffold training molecules:', len(sc_train_idx))
print('Scaffold test molecules:', len(sc_test_idx))
print('Training scaffold groups:', len(set(scaffolds[sc_train_idx])))
print('Testing scaffold groups:', len(set(scaffolds[sc_test_idx])))
print('Overlapping scaffold groups: 0')

scaffold_rf = RandomForestRegressor(n_estimators=200, max_features='sqrt',
    min_samples_leaf=2, random_state=SEED, n_jobs=-1)
t0 = time.time()
scaffold_rf.fit(X[sc_train_idx], y[sc_train_idx])
scaffold_predictions = scaffold_rf.predict(X[sc_test_idx])
scaffold_metrics = metrics('Scaffold-group Random Forest', y[sc_test_idx], scaffold_predictions)
print('Scaffold training seconds:', round(time.time()-t0, 1))
display(pd.DataFrame([scaffold_metrics]).round(4))


### Comparison
The random and scaffold experiments use different training sizes and different test molecules; differences cannot be attributed solely to scaffold novelty. A single split is insufficient to quantify variability across seeds.

In [ ]:
split_comparison = pd.DataFrame([
    {'Split':'Random', **metrics('RF', y_test, rf_predictions)},
    {'Split':'Scaffold-group', **scaffold_metrics}])
display(split_comparison[['Split', 'MAE', 'RMSE', 'R2']].round(4))
split_comparison.to_csv(OUT / 'split_comparison.csv', index=False)
joblib.dump(scaffold_rf, OUT / 'scaffold_random_forest.joblib')


## 6. All-molecule predictions and held-out error analysis
We can predict all 9,980 structures, but the training-partition predictions are **not independent validation**. Identify test rows using the saved scaffold split indices.

In [ ]:
all_results = pd.DataFrame({
    'SMILES': df['smiles'], 'Experimental_logS': y,
    'Predicted_logS': scaffold_rf.predict(X)})
labels = np.full(len(df), 'Training', dtype=object)  # Prevent string truncation
labels[sc_test_idx] = 'Held-out Test'
all_results['Split'] = labels
all_results['Signed_Error'] = all_results['Predicted_logS'] - all_results['Experimental_logS']
all_results['Absolute_Error'] = all_results['Signed_Error'].abs()
heldout = all_results.loc[all_results['Split'] == 'Held-out Test'].copy()
assert len(heldout) == len(sc_test_idx)
assert np.isclose(heldout['Absolute_Error'].mean(), scaffold_metrics['MAE'])
print('All molecules:', len(all_results), '| held-out:', len(heldout))
print('Held-out MAE:', round(heldout['Absolute_Error'].mean(), 4))
all_results.to_csv(OUT / 'all_predictions.csv', index=False)
worst_10 = heldout.nlargest(10, 'Absolute_Error').copy()
display(worst_10[['SMILES','Experimental_logS','Predicted_logS','Absolute_Error']].round(3))
worst_10.to_csv(OUT / 'top_10_errors.csv', index=False)


### Lipophilicity and error
RDKit `MolLogP` estimates lipophilicity; it is **not** the target logS. Compare prediction errors across LogP strata using the **entire scaffold-held-out set**, not only the worst ten. Group associations are exploratory, not causal.

In [ ]:
heldout['LogP'] = [Descriptors.MolLogP(molecules[i]) for i in sc_test_idx]
heldout['MolecularWeight'] = [Descriptors.MolWt(molecules[i]) for i in sc_test_idx]
heldout['LogP_Group'] = pd.cut(heldout['LogP'],
    [-np.inf, 2, 4, np.inf], labels=['Low (< 2)', 'Medium (2–4)', 'High (> 4)'])
logp_summary = heldout.groupby('LogP_Group', observed=True).agg(
    Molecules=('Absolute_Error', 'size'), MAE=('Absolute_Error', 'mean'))
display(logp_summary.round(3))
logp_summary.to_csv(OUT / 'logp_error_analysis.csv')
ax = logp_summary['MAE'].plot.bar(rot=0, figsize=(7,4), title='Scaffold-held-out MAE by LogP')
ax.set_ylabel('MAE (logS)'); plt.tight_layout()
plt.savefig(OUT / 'logp_error_analysis.png', dpi=180); plt.show()
heldout.nlargest(10, 'Absolute_Error').to_csv(OUT / 'chemical_error_analysis.csv', index=False)


## 7. Inference on example molecules
Use exactly the same RDKit fingerprint settings used in training. Example predictions are demonstrations, **not external test results**; some compounds may already be present in AqSolDB.

In [ ]:
def predict_solubility(smiles, model=scaffold_rf):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError('Invalid SMILES')
    fp = fpgen.GetFingerprintAsNumPy(mol).reshape(1, -1)
    return float(model.predict(fp)[0])

examples = {
    'Aspirin': 'CC(=O)Oc1ccccc1C(=O)O',
    'Caffeine': 'Cn1c(=O)c2c(ncn2C)n(C)c1=O',
    'Paracetamol': 'CC(=O)Nc1ccc(O)cc1',
}
example_predictions = pd.DataFrame([
    {'Molecule': name, 'SMILES': smi,
     'Predicted_logS': predict_solubility(smi)}
    for name, smi in examples.items()])
example_predictions['Predicted_mol_per_L'] = 10 ** example_predictions['Predicted_logS']
display(example_predictions.round(5))
example_predictions.to_csv(OUT / 'example_predictions.csv', index=False)


## 8. Conclusions and limitations

**Previously observed original experiment:** Random split MAE 1.1357, R² 0.6108; scaffold-group split MAE 1.4686, R² 0.5091. The scaffold-group split included 8,760 training and 1,220 test molecules and zero scaffold overlap. In the original held-out error analysis, LogP > 4 compounds had MAE 2.203 versus 1.095 for LogP 2–4.

**Limitations:** Only one random seed/split; train and test distribution differences; hashing collisions; basic SMILES canonicalization rather than complete standardization; potential inconsistencies in experimental solubility conditions; no external validation or calibrated uncertainty. A model's error associations with LogP do not establish causation. Different RDKit/scikit-learn versions may slightly change results.

**Next research directions:** Repeated scaffold splits, controlled descriptor-augmented models (LogP, TPSA, molecular weight), applicability-domain analysis, experimental validation.

**License/attribution:** Cite the curated AqSolDB source and comply with dataset terms. MIT (if chosen) applies to original project code, not automatically to source data.